In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier
from sklearn.metrics import accuracy_score , precision_score , recall_score,f1_score
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import dagshub
dagshub.init(repo_owner='Aayush10671', repo_name='emotion-detection', mlflow=True)

mlflow.set_tracking_uri('https://dagshub.com/Aayush10671/emotion-detection.mlflow')


Accessing as Aayush10671

Initialized MLflow to track repo "Aayush10671/emotion-detection"

Repository Aayush10671/emotion-detection initialized!

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv')
df.head(5)

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


In [4]:
# data preprocessing



def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)


def normalize_text(df):
    """Normalize the text data."""
    try:
        # Use 'content' instead of 'text' since that's the actual column name
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        raise
  

In [5]:
df = normalize_text(df)
df.head(5)

,tweet_id,sentiment,content
0,1956967341,empty,tiffanylue know listenin bad habit earlier sta...
1,1956967666,sadness,layin n bed headache ughhhh waitin call
2,1956967696,sadness,funeral ceremony gloomy friday
3,1956967789,enthusiasm,want hang friend soon
4,1956968416,neutral,dannycastillo want trade someone houston ticke...


In [6]:
X = df['sentiment'].isin(['happiness' , 'sadness'])
df = df[X]

In [7]:
df['sentiment'] = df['sentiment'].replace({'happiness':1 , 'sadness':0})

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_9732\3399834364.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'happiness':1 , 'sadness':0})


In [9]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['content'])
y = df['sentiment']

In [10]:
X_train , X_test , y_train, y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [11]:
mlflow.set_experiment("LogisticRegression hyperparameter Tuning")

2026/07/17 12:14:42 INFO mlflow.tracking.fluent: Experiment with name 'LogisticRegression hyperparameter Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/fafacfe0478840bc8cb58c789aa0b9d4', creation_time=1784270682287, experiment_id='2', last_update_time=1784270682287, lifecycle_stage='active', name='LogisticRegression hyperparameter Tuning', tags={}, workspace='default'>

In [15]:
param_grid = {
    'C' : [0.1 , 1 , 10],
    'penalty' : ['l1','l2'],
    'solver' : ['liblinear']
}

In [16]:

from sklearn.model_selection import GridSearchCV

with mlflow.start_run():
    # Create GridSearchCV
    grid_search = GridSearchCV(
        LogisticRegression(random_state=42, max_iter=1000),
        param_grid,
        cv=5,
        scoring='f1',
        n_jobs=-1
    )
    
    # Fit the model
    grid_search.fit(X_train, y_train)
    
    # Loop through each parameter combination
    for i in range(len(grid_search.cv_results_['params'])):
        params = grid_search.cv_results_['params'][i]
        mean_cv_score = grid_search.cv_results_['mean_test_score'][i]
        std_cv_score = grid_search.cv_results_['std_test_score'][i]
        
        # Train model with these params to get test metrics
        model = LogisticRegression(**params, max_iter=1000, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        # Log each model in a nested run
        with mlflow.start_run(nested=True, run_name=f"model_{i}"):
            mlflow.log_params(params)
            mlflow.log_metric('cv_mean_f1', mean_cv_score)
            mlflow.log_metric('cv_std_f1', std_cv_score)
            mlflow.log_metric('test_accuracy', accuracy)
            mlflow.log_metric('test_precision', precision)
            mlflow.log_metric('test_recall', recall)
            mlflow.log_metric('test_f1', f1)
            mlflow.sklearn.log_model(model, f"model_{i}")
            
            # Print metrics for this model
            print(f"\n{'='*60}")
            print(f"Model {i+1} with params: {params}")
            print(f"{'='*60}")
            print(f"CV Mean F1 Score: {mean_cv_score:.4f} (±{std_cv_score:.4f})")
            print(f"Test Accuracy:     {accuracy:.4f}")
            print(f"Test Precision:    {precision:.4f}")
            print(f"Test Recall:       {recall:.4f}")
            print(f"Test F1 Score:     {f1:.4f}")
    
    # Get best model and its metrics
    best_params = grid_search.best_params_
    best_cv_score = grid_search.best_score_
    best_model = grid_search.best_estimator_
    
    # Predict with best model
    y_pred_best = best_model.predict(X_test)
    best_accuracy = accuracy_score(y_test, y_pred_best)
    best_precision = precision_score(y_test, y_pred_best, average='weighted')
    best_recall = recall_score(y_test, y_pred_best, average='weighted')
    best_f1 = f1_score(y_test, y_pred_best, average='weighted')
    
    # Log best model in main run
    mlflow.log_params(best_params)
    mlflow.log_metric('best_cv_f1', best_cv_score)
    mlflow.log_metric('best_test_accuracy', best_accuracy)
    mlflow.log_metric('best_test_precision', best_precision)
    mlflow.log_metric('best_test_recall', best_recall)
    mlflow.log_metric('best_test_f1', best_f1)
    mlflow.sklearn.log_model(best_model, "best_model")
    
    # Print best model summary
    print(f"\n{'='*60}")
    print(f"🏆 BEST MODEL (Selected by GridSearchCV)")
    print(f"{'='*60}")
    print(f"Best Parameters: {best_params}")
    print(f"Best CV F1 Score: {best_cv_score:.4f}")
    print(f"\nTest Set Performance of Best Model:")
    print(f"  Accuracy:  {best_accuracy:.4f}")
    print(f"  Precision: {best_precision:.4f}")
    print(f"  Recall:    {best_recall:.4f}")
    print(f"  F1 Score:  {best_f1:.4f}")
    

2026/07/17 12:30:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/17 12:30:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Model 1 with params: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}
CV Mean F1 Score: 0.7052 (±0.0142)
Test Accuracy:     0.7398
Test Precision:    0.7442
Test Recall:       0.7398
Test F1 Score:     0.7379
🏃 View run model_0 at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/2/runs/6d0b9131f3ed4e729db4828419033a05
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/2


2026/07/17 12:30:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/17 12:30:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Model 2 with params: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
CV Mean F1 Score: 0.7792 (±0.0107)
Test Accuracy:     0.7894
Test Precision:    0.7895
Test Recall:       0.7894
Test F1 Score:     0.7894
🏃 View run model_1 at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/2/runs/25acfb87ef4241da9b2fd9342c5c1790
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/2


2026/07/17 12:30:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/17 12:30:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...


Model 5 with params: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
CV Mean F1 Score: 0.7716 (±0.0117)
Test Accuracy:     0.7836
Test Precision:    0.7836
Test Recall:       0.7836
Test F1 Score:     0.7836
🏃 View run model_4 at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/2/runs/f2f0cfeb7e794ff6acc0a2b5644aa43c
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/2


2026/07/17 12:32:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/17 12:32:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Model 6 with params: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
CV Mean F1 Score: 0.7792 (±0.0047)
Test Accuracy:     0.7807
Test Precision:    0.7809
Test Recall:       0.7807
Test F1 Score:     0.7807
🏃 View run model_5 at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/2/runs/9c2d3d116ec54469b2588490c5b67041
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/2


2026/07/17 12:33:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/17 12:33:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏆 BEST MODEL (Selected by GridSearchCV)
Best Parameters: {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}
Best CV F1 Score: 0.7904

Test Set Performance of Best Model:
  Accuracy:  0.7952
  Precision: 0.7953
  Recall:    0.7952
  F1 Score:  0.7952
🏃 View run bemused-penguin-688 at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/2/runs/cfa3010b503b4207b0ae32427afb0929
🧪 View experiment at: https://dagshub.com/Aayush10671/emotion-detection.mlflow/#/experiments/2
